# Lichess Matchmaking EDA

This notebook explores the public Lichess games dataset as a proxy for 1v1 competitive matchmaking. The goal is to understand rating balance, outcome predictability, time-control behavior, and practical signals for a matchmaking policy.

## Dataset

- Kaggle: https://www.kaggle.com/datasets/datasnaek/chess
- Zenodo mirror: https://zenodo.org/records/5978831
- Direct CSV: https://zenodo.org/records/5978831/files/games.csv?download=1

Run `python scripts/download_data.py` from the repository root before executing this notebook.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.append(str(ROOT / 'src'))

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from matchmaking_system.data import load_games, prepare_match_records
from matchmaking_system.features import add_matchmaking_features

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 100)

In [ ]:
raw = load_games(ROOT / 'data/raw/games.csv')
raw.shape, raw.head()

In [ ]:
raw.info()

In [ ]:
raw.isna().mean().sort_values(ascending=False).head(10)

## Match Balance

A fair 1v1 match should have a small skill gap and an expected win probability close to 50%.

In [ ]:
matches = prepare_match_records(raw, include_draws=True)
matches = add_matchmaking_features(matches)
matches[['white_rating', 'black_rating', 'rating_gap', 'abs_rating_gap', 'elo_white_win_prob', 'elo_uncertainty']].describe()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.histplot(matches['abs_rating_gap'], bins=40, ax=axes[0])
axes[0].set_title('Absolute rating gap')
axes[0].set_xlabel('Rating points')

sns.histplot(matches['elo_white_win_prob'], bins=40, ax=axes[1])
axes[1].axvline(0.5, color='black', linestyle='--', linewidth=1)
axes[1].set_title('Elo expected white win probability')
axes[1].set_xlabel('Probability')
plt.tight_layout()

## Outcome Patterns

We compare actual outcomes with rating-derived expectations. This is the bridge from EDA to an ML-assisted matchmaking policy.

In [ ]:
matches['winner'].value_counts(normalize=True).rename('share').to_frame()

In [ ]:
binary = prepare_match_records(raw, include_draws=False)
binary = add_matchmaking_features(binary)
binary['rating_gap_bin'] = pd.cut(binary['rating_gap'], bins=[-1000, -400, -200, -100, 0, 100, 200, 400, 1000])
calibration = binary.groupby('rating_gap_bin', observed=True).agg(
    games=('white_win', 'size'),
    actual_white_win_rate=('white_win', 'mean'),
    elo_expected_white_win_rate=('elo_white_win_prob', 'mean'),
    avg_abs_rating_gap=('abs_rating_gap', 'mean'),
).reset_index()
calibration

In [ ]:
plt.figure(figsize=(10, 5))
plot_df = calibration.melt(
    id_vars='rating_gap_bin',
    value_vars=['actual_white_win_rate', 'elo_expected_white_win_rate'],
    var_name='metric',
    value_name='rate',
)
sns.lineplot(data=plot_df, x=plot_df['rating_gap_bin'].astype(str), y='rate', hue='metric', marker='o')
plt.xticks(rotation=45, ha='right')
plt.ylim(0, 1)
plt.title('Actual vs Elo-expected outcome by rating gap')
plt.tight_layout()

## Time Control and Engagement Proxies

The dataset does not include satisfaction or retention directly. We use game length, decisive outcomes, and timeout/resignation status as portfolio-friendly proxies for match experience.

In [ ]:
time_control_summary = matches.groupby('increment_code').agg(
    games=('id', 'size'),
    avg_turns=('turns', 'mean'),
    median_abs_rating_gap=('abs_rating_gap', 'median'),
    avg_uncertainty=('elo_uncertainty', 'mean'),
).sort_values('games', ascending=False).head(15)
time_control_summary

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.boxplot(data=matches, x='winner', y='abs_rating_gap', ax=axes[0])
axes[0].set_title('Rating gap by outcome')

top_statuses = matches['victory_status'].value_counts().index
sns.boxplot(data=matches[matches['victory_status'].isin(top_statuses)], x='victory_status', y='turns', ax=axes[1])
axes[1].set_title('Game length by victory status')
plt.tight_layout()

## EDA Findings to Carry Forward

- Rating gap is the core fairness signal and should be a hard or heavily weighted constraint.
- Elo expected probability creates an interpretable baseline for win-probability calibration.
- Time-control and game-length features are useful context for engagement-aware matching.
- Draws should be kept for fairness analysis but filtered or modeled separately for binary win prediction.
- A portfolio-ready solution should compare policies offline before proposing online A/B testing.